In [1]:
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
df = pd.read_excel("salary_data_v2.xlsx")
df = df.drop(columns=["Employee ID","Email","Random Code","Joining Date"], errors="ignore")
 

In [3]:
for col in ["Education Degree","Specialization","Education Level","Job Role","Company Size"]:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in ["Education Degree","Specialization","Education Level","Job Sector","Job Role","Company Size","City"]:
    if col in df.columns:
        df[col] = df[col].str.strip().str.title()

tier1 = ["Bengaluru","Mumbai","Delhi Ncr","Hyderabad","Chennai","Pune"]
tier2 = ["Noida","Gurgaon","Ahmedabad","Kolkata"]
df["City Tier"] = df["City"].apply(
    lambda c: "Tier 1 (Metro)" if c in tier1 else ("Tier 2 (Mid-Sized)" if c in tier2 else "Tier 3 (Smaller Cities)")
    )

In [4]:
FEATURES  = ["Experience (Years)","Education Degree","Specialization","Education Level","Job Sector","Job Role","Company Size","City Tier"]
cat_feats = ["Education Degree","Specialization","Education Level","Job Sector","Job Role","Company Size","City Tier"]
num_feats = ["Experience (Years)"]

X = df[FEATURES]
y = df["Salary (INR)"]

In [5]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_feats),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), cat_feats),
    ])

In [6]:
pipeline = Pipeline([
        ("pre", preprocessor),
        ("reg", GradientBoostingRegressor(n_estimators=200, learning_rate=0.08,
                                          max_depth=5, random_state=42)),
    ])


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('pre', ...), ('reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Experience (Years)','Education Degree','Specialization',...,'Job Role', 'Company Size','City Tier']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This 

In [8]:
y_pred = pipeline.predict(X_test)
print(mean_squared_error(y_test, y_pred))
print(r2_score(y_test, y_pred))

109844523710.09059
0.9223672977581183


In [9]:
# Save trained pipeline
joblib.dump(pipeline, "models/salary_model.pkl")


['models/salary_model.pkl']